# 6.2 - The Stable Diffusion Ecosystem

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook takes the open Stable Diffusion pipeline apart and rebuilds it with the `diffusers` library: you load a checkpoint and count its three sub-networks, generate an image and turn each control dial, edit and inpaint existing images, then customize with a LoRA and speed the whole thing up for production. Every cell is a small, runnable piece of a shippable image pipeline rather than a black-box API call.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Installs the Hugging Face image-generation stack. `torch` ships with Colab already; `diffusers` (the pipelines), `transformers` (the CLIP text encoders), and `accelerate` (device placement / fast loading) do not, so we add them.

> **Needs a GPU** - every generation cell below calls `.to("cuda")`; shown for reference if you have no NVIDIA GPU (use `.to("cpu")`, far slower).

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# torch is preinstalled in Colab; diffusers/transformers/accelerate are not:
!pip install diffusers transformers accelerate -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Pure environment prep - a `pip install`, not a model call. It pulls the three libraries the rest of the notebook imports so the SDXL pipeline can be assembled on top of the preinstalled `torch`.

**How the code works - step by step**
- **`!pip install diffusers transformers accelerate -q`** - `diffusers` supplies the pipeline classes, `transformers` the CLIP text encoders inside them, and `accelerate` the device/loading helpers; `-q` keeps the log quiet.
- **`torch`** is assumed already present (Colab preinstalls it), so it is deliberately not in the install line.

**In one line:** install the diffusers stack on top of the GPU's existing torch.

**What you'll see:** (no output - environment prep)

## 1 - Take the pipeline apart and count its three parts

A Stable Diffusion "checkpoint" is not one model - it is a text encoder, a denoiser, and a VAE, plus a scheduler, bundled together. This cell loads SDXL and counts the parameters of each sub-network so you can see the three separately trained parts with your own eyes.

> **Needs a GPU** - loads SDXL in half precision; the count itself runs on CPU but the download is multi-GB.

In [ ]:
# pip install diffusers transformers accelerate torch
from diffusers import StableDiffusionXLPipeline
import torch

# A "checkpoint" bundles all three components into one pipeline object.
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,   # 16-bit half precision (~half the GPU memory)
)

# Peek inside - the three separately trained parts:
def count(m):
    return sum(p.numel() for p in m.parameters())

print(f"text encoder(s): {count(pipe.text_encoder) + count(pipe.text_encoder_2):,} params")
print(f"denoiser (UNet): {count(pipe.unet):,} params")
print(f"VAE:             {count(pipe.vae):,} params")
print(f"scheduler:       {type(pipe.scheduler).__name__}")
# Output: text encoder(s): 817,785,409 params   (SDXL has TWO CLIP encoders)
# Output: denoiser (UNet): 2,567,463,684 params  (the big one)
# Output: VAE:             83,653,863 params
# Output: scheduler:       EulerDiscreteScheduler

**Explanation**

This is an inspection cell, not a generation cell - it loads the pipeline and then reaches inside it to weigh each component. The point is to make concrete that a checkpoint bundles three replaceable networks, which is the foundation for swapping and customizing them later in the notebook.

**How the code works - step by step**
- **`StableDiffusionXLPipeline.from_pretrained(...)`** - downloads and assembles all three components plus a scheduler into one `pipe` object; `torch_dtype=torch.float16` loads them in 16-bit half precision (~half the VRAM).
- **`count(m)`** - a helper that sums `p.numel()` over a module's parameters, i.e. counts its learned numbers.
- **text encoder(s)** - `pipe.text_encoder + pipe.text_encoder_2`; SDXL runs two CLIP encoders, which is why both are added.
- **denoiser (`pipe.unet`)** - the biggest part by far, the noise-predictor from Lesson 6.1, where LoRAs and ControlNet attach.
- **VAE (`pipe.vae`)** - the smallest, compresses images to latents and decodes them back to pixels.
- **`type(pipe.scheduler).__name__`** - prints the sampler class (diffusers calls the sampler a "scheduler").

**In one line:** load the checkpoint, then weigh each of its three networks to prove it is three parts, not one.

**What you'll see:** Four printed lines: text encoder(s) ~817M params, denoiser (UNet) ~2.57B params (the dominant one), VAE ~83.6M params, and scheduler `EulerDiscreteScheduler`.

## 2 - Generate an image and turn the control dials

Text-to-image is only five lines - the skill is the handful of dials that decide whether the image is rough, right, or ruined. This cell renders one prompt and sets every dial explicitly so you can see what each one does.

> **Needs a GPU** - full 1024x1024 SDXL generation.

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
).to("cuda")

# swap the sampler (scheduler) to trade speed/quality, e.g.:
# from diffusers import DPMSolverMultistepScheduler
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

image = pipe(
    prompt="a red origami crane on a wooden desk, soft window light, photographic",
    negative_prompt="blurry, distorted, extra creases, low quality",  # steer AWAY from these
    num_inference_steps=28,     # sharpness; ~25-30 is the usual range
    guidance_scale=7.0,         # prompt adherence (CFG from 6.1); SDXL sweet spot ~5-8
    width=1024, height=1024,   # SDXL native resolution
    generator=torch.Generator("cuda").manual_seed(42),  # fixed seed = reproducible
).images[0]
image.save("crane.png")
print(image.size)
# Output: (1024, 1024)

**Explanation**

A minimal end-to-end generation call with every knob named. Read it as one `pipe(...)` invocation whose keyword arguments are the control surface: steps for sharpness, guidance for prompt adherence, negative prompt for what to avoid, and a seeded generator for reproducibility.

**How the code works - step by step**
- **`from_pretrained(...).to("cuda")`** - same SDXL checkpoint as Step 1, now moved onto the GPU to run.
- **`prompt` / `negative_prompt`** - what to aim for, and what to steer away from (`blurry, distorted, ...`) via the unconditioned pass - a cheap quality lever.
- **`num_inference_steps=28`** - denoising passes; sharpness, with diminishing returns past ~30.
- **`guidance_scale=7.0`** - the CFG dial (prompt adherence); SDXL's sweet spot is ~5-8 (SD3.5/Flux want lower).
- **`width=1024, height=1024`** - SDXL's native resolution.
- **`generator=torch.Generator("cuda").manual_seed(42)`** - fixes the starting noise so the run is reproducible; change it for a different image of the same prompt.
- **commented `DPMSolverMultistepScheduler`** - shows how to swap the sampler to trade speed for quality without blindly cutting steps.
- **`.images[0]` / `image.save(...)`** - the call returns a list; grab the first image and write it to disk.

**In one line:** one `pipe()` call, and each keyword is a dial with its own sweet spot.

**What you'll see:** Saves `crane.png` and prints the image size `(1024, 1024)`.

## 3 - Edit instead of create: img2img and inpainting

Same pipeline, different starting point - feed in an existing image, noise it partway, and denoise toward a new prompt. `strength` sets how much of the original you keep; a mask lets you edit one region and copy the rest verbatim.

> **Needs a GPU** - loads two SDXL checkpoints and downloads sample assets over the network.

In [ ]:
from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
from diffusers.utils import load_image
import torch

img2img = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# Hosted sample image + matching mask (diffusers inpainting example assets).
_img_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
_mask_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo_mask.png"
init = load_image(_img_url).resize((1024, 1024))

# strength controls how much of the input is noised away:
edit = img2img(
    prompt="the same desk at golden hour, warm cinematic light",
    image=init,
    strength=0.45,        # 0 = keep input, 1 = ignore it; 0.3-0.6 edits
    guidance_scale=7.0,
).images[0]
edit.save("edited.png")
print("img2img done")
# Output: img2img done

# INPAINTING: edit ONLY a masked region, copy the rest verbatim.
# Note: inpainting needs a checkpoint fine-tuned for masking - the plain SDXL
# base id won't work here, which is why the model id below is different.
inpaint = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16
).to("cuda")

mask = load_image(_mask_url).resize((1024, 1024))   # white = edit here, black = keep
fixed = inpaint(
    prompt="a ceramic coffee mug", image=init, mask_image=mask, strength=0.85
).images[0]
fixed.save("inpainted.png")
print("inpaint done")
# Output: inpaint done

**Explanation**

Two related edits in one cell. Image-to-image reuses the base checkpoint but starts denoising from your noised input; inpainting swaps in a mask-fine-tuned checkpoint and a mask so only white pixels change. The single idea tying them together is `strength` - how much noise is added before the redraw.

**How the code works - step by step**
- **`AutoPipelineForImage2Image.from_pretrained(...)`** - loads SDXL as an img2img pipeline; `AutoPipelineFor*` picks the right class so only the model id changes across model families.
- **`load_image(_img_url).resize((1024,1024))`** - pulls a hosted sample image (and later its matching mask) and sizes it to SDXL's resolution.
- **`img2img(..., strength=0.45)`** - noises the input ~45% away before denoising to the golden-hour prompt: low keeps the image, ~0.45 edits it, near 1.0 regenerates it.
- **`AutoPipelineForInpainting.from_pretrained("diffusers/stable-diffusion-xl-1.0-inpainting-0.1")`** - a *different*, mask-fine-tuned checkpoint (plain SDXL base won't inpaint).
- **`inpaint(..., mask_image=mask, strength=0.85)`** - white mask pixels are regenerated to "a ceramic coffee mug", black pixels copied verbatim; high strength gives a near-full redraw *inside* the mask while the mask protects everything outside.
- **`.save(...)`** - writes `edited.png` and `inpainted.png`.

**In one line:** noise the input partway (strength), and optionally let a mask decide which pixels get redrawn.

**What you'll see:** Prints `img2img done` then `inpaint done`, and saves `edited.png` and `inpainted.png`.

## 4 - Customize with a LoRA adapter

Teach a base model a style, character, or product with a file smaller than a photo. A LoRA is a tiny adapter that clips onto the frozen base - swap it, dial its strength, or stack two - without ever retraining the multi-GB model.

> **Needs a GPU** - loads SDXL plus a LoRA and generates an image.

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# Load a LoRA adapter (a 4-20 MB .safetensors file - the standard safe-to-load
# weight format - e.g. from Civitai). Name it so we can reference it below.
pipe.load_lora_weights("TheLastBen/Papercut_SDXL", weight_name="papercut.safetensors",
                       adapter_name="cutout")
pipe.set_adapters(["cutout"], adapter_weights=[0.8])   # dial the strength

# Stack a second LoRA (e.g. a character) by naming each and listing both:
# pipe.load_lora_weights("./loras", weight_name="my-character.safetensors",
#                        adapter_name="char")
# pipe.set_adapters(["cutout", "char"], adapter_weights=[0.8, 0.7])

image = pipe(
    prompt="a fox in a forest, paper-cutout style",  # trigger word from the LoRA
    num_inference_steps=28, guidance_scale=7.0,
).images[0]
image.save("fox.png")
print("LoRA applied - base weights never changed")
# Output: LoRA applied - base weights never changed

**Explanation**

Shows the consume-a-LoRA path (loading, not training). The base pipeline stays frozen; a few-MB `.safetensors` adapter is attached, activated by name, and given a strength weight. The commented lines demonstrate stacking a second adapter on top.

**How the code works - step by step**
- **`from_pretrained(...).to("cuda")`** - the same frozen SDXL base as before.
- **`load_lora_weights("TheLastBen/Papercut_SDXL", ..., adapter_name="cutout")`** - downloads and attaches a 4-20 MB paper-cutout adapter, named so it can be referenced.
- **`set_adapters(["cutout"], adapter_weights=[0.8])`** - activates the adapter and dials how strongly it applies (0.8).
- **commented stack block** - load a second `char` adapter and pass both names + both weights to blend a style with a character.
- **`pipe(prompt="... paper-cutout style", ...)`** - the prompt includes the LoRA's trigger word to fire the learned concept; steps/guidance are the usual SDXL values.
- **`.save("fox.png")`** - writes the restyled image.

**In one line:** clip a small adapter onto the frozen base, name it, weight it, and trigger it in the prompt.

**What you'll see:** Saves `fox.png` and prints `LoRA applied - base weights never changed`.

## 5 - Ship it: few-step speed and a safety gate

Going from a notebook to a service means making generation cheap and safe. This cell loads an LCM-LoRA that turns any SDXL into a ~4-step model - the single biggest cost lever - and wraps the output in a safety check you must never ship without.

> **Needs a GPU** - loads SDXL plus the LCM-LoRA and generates.

In [ ]:
from diffusers import StableDiffusionXLPipeline, LCMScheduler
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# SPEED: an LCM-LoRA turns any SDXL into a ~4-step model.
pipe.load_lora_weights("latent-consistency/lcm-lora-sdxl")  # a single LoRA is active on load - no set_adapters needed
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

image = pipe(
    prompt="a ceramic mug on a marble counter, product photo",
    num_inference_steps=4,      # 4 instead of 28 -> ~7x less compute
    guidance_scale=1.5,         # distilled models need very low guidance (see note)
).images[0]

# SAFETY: never ship without an output check (deepfake / NSFW / brand risk).
def passes_safety(img) -> bool:
    # plug in your classifier / provider safety checker here
    return True

print("ok" if passes_safety(image) else "blocked")
# Output: ok

**Explanation**

A production-shaped generation: a distillation adapter plus a matching scheduler collapse ~28 steps to ~4, and a `passes_safety` gate stands between the model output and shipping. The safety function is a stub you replace with a real classifier - the pattern, not the check, is the point.

**How the code works - step by step**
- **`from_pretrained(...).to("cuda")`** - the SDXL base again.
- **`load_lora_weights("latent-consistency/lcm-lora-sdxl")`** - a single LCM (Latent Consistency Model) adapter; because only one LoRA is active on load, no `set_adapters` call is needed.
- **`pipe.scheduler = LCMScheduler.from_config(...)`** - swaps in the scheduler the few-step method requires.
- **`num_inference_steps=4`** - 4 instead of 28, ~7x less compute.
- **`guidance_scale=1.5`** - distilled models need very low guidance regardless of base model - a property of the distillation, not a contradiction of Step 2's ~5-8.
- **`passes_safety(img)`** - a placeholder returning `True`; plug your NSFW / deepfake / brand classifier here.
- **`print("ok" if ... else "blocked")`** - gates the output before it ships.

**In one line:** distill to 4 steps at low guidance, then never ship without a safety check.

**What you'll see:** Prints `ok` (the stubbed safety check always passes).

Across six cells you moved from opening a checkpoint to shipping one: you saw a Stable Diffusion pipeline is three trained networks plus a scheduler, turned the text-to-image dials (steps, guidance, seed, negative prompt, sampler), edited images with img2img strength and mask-based inpainting, clipped on a LoRA adapter without touching the frozen base, and cut a 28-step render to 4 with an LCM-LoRA behind a safety gate. The through-line is that openness buys control at every stage. Next, Lesson 6.3 adds structural control (ControlNet, IP-Adapter) so you can impose exact pose, depth, and edges on top of everything here.